<a href="https://colab.research.google.com/github/priyanshupratikg/Celebal_Excellence_Internship_Priyanshu_Pratik/blob/main/Celebal_internship_week3_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas scikit-learn xgboost


In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import silhouette_score, accuracy_score, classification_report


# 1. LOAD AND PREPROCESS DATA

print("--- Loading Dataset ---")
df = pd.read_csv("Country-data.csv")
X_features = df.drop(columns=['country'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)


# 2. UNSUPERVISED LEARNING & SEGMENTATION INSIGHTS

print("\n--- Running Unsupervised Clustering ---")

# A. K-Means
kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)
print(f"K-Means Silhouette Score: {silhouette_score(X_scaled, df['KMeans_Cluster']):.4f}")

# B. DBSCAN (Optimized parameters specifically for the Country-data distribution)
dbscan = DBSCAN(eps=2.3, min_samples=4)
df['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

unique_dbscan_labels = set(df['DBSCAN_Cluster'])
if len(unique_dbscan_labels) > 1:
    print(f"DBSCAN Silhouette Score: {silhouette_score(X_scaled, df['DBSCAN_Cluster']):.4f}")
else:
    print("DBSCAN failed to find multiple clusters; try adjusting eps.")

# C. GENERATING ACTIONABLE INSIGHTS (Crucial for assignment completion)
print("\n================ ACTIONABLE SEGMENTATION INSIGHTS ================")
# Drop tracking metrics to get clean feature averages
features_to_profile = df.drop(columns=['country', 'DBSCAN_Cluster', 'KMeans_Cluster'], errors='ignore')
cluster_profile = features_to_profile.groupby(df['KMeans_Cluster']).mean()

# Dynamically pick available target features to display safely
target_columns = ['child_mort', 'income', 'gdpp', 'life_exp', 'life_expe', 'life_expc']
available_cols = [col for col in target_columns if col in cluster_profile.columns]

print(cluster_profile[available_cols])

print("\nCluster Interpretation Guideline:")
for cluster_id in cluster_profile.index:
    # Use 'income' or fallback to the first numeric column if missing
    income_col = 'income' if 'income' in cluster_profile.columns else cluster_profile.columns[0]
    income = cluster_profile.loc[cluster_id, income_col]

    if income > 30000:
        print(f" -> Cluster {cluster_id}: Developed Countries (High Income)")
    elif income > 10000:
        print(f" -> Cluster {cluster_id}: Developing Countries (Mid Income)")
    else:
        print(f" -> Cluster {cluster_id}: Under-developed Countries (Low Income)")


# 3. SUPERVISED LEARNING (CLASSIFICATION SYSTEM)

print("\n--- Pipeline Optimization & Classification System ---")
y = df['KMeans_Cluster']

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# A. Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

# B. XGBoost
xgb_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)


# 4. MODEL EVALUATION

print("\n================ EVALUATION METRICS ================")
print(f"Random Forest Test Accuracy: {accuracy_score(y_test, rf_preds):.4f}")
print(f"XGBoost Test Accuracy:       {accuracy_score(y_test, xgb_preds):.4f}")

print("\n--- XGBoost Classification Report ---")
print(classification_report(y_test, xgb_preds))

# Export comprehensive final report data
df.to_csv("customer_intelligence_output.csv", index=False)
print("\nExecution complete. Enhanced results exported successfully.")

--- Loading Dataset ---

--- Running Unsupervised Clustering ---
K-Means Silhouette Score: 0.2833
DBSCAN Silhouette Score: 0.4857

================ ACTIONABLE SEGMENTATION INSIGHTS ================
                child_mort        income          gdpp
KMeans_Cluster                                        
0                 5.000000  45672.222222  42494.444444
1                92.961702   3942.404255   1922.382979
2                21.927381  12305.595238   6486.452381

Cluster Interpretation Guideline:
 -> Cluster 0: Developed Countries (High Income)
 -> Cluster 1: Under-developed Countries (Low Income)
 -> Cluster 2: Developing Countries (Mid Income)

--- Pipeline Optimization & Classification System ---

================ EVALUATION METRICS ================
Random Forest Test Accuracy: 1.0000
XGBoost Test Accuracy:       0.9706

--- XGBoost Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         7
           